In [1]:
!pip install -q langgraph langchain-huggingface langchain-core

In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
import os
from google.colab import userdata

In [3]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')

llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    provider="cerebras",   # plain chat, no structured output -> cerebras is fine
    task="text-generation"
)
model = ChatHuggingFace(llm=llm_endpoint)

In [4]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    # f-string here too, for the same reason as the single-node file: this is
    # plain Python inside a node, not an LCEL chain, so no PromptTemplate needed
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']   # <-- reads what node 1 wrote into state

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [7]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')   # <-- THIS edge is what makes it a pipeline
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [8]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

Our latest automated health check on model 'openai/gpt-oss-120b' for provider 'cerebras' did not complete successfully.  Inference call might fail.
Our latest automated health check on model 'openai/gpt-oss-120b' for provider 'cerebras' did not complete successfully.  Inference call might fail.


{'title': 'Rise of AI in India', 'outline': '**Blog Title:**  \n*The Rise of Artificial Intelligence in India: Opportunities, Challenges, and the Road Ahead*\n\n---\n\n## 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “India is projected to host 2.2\u202fmillion AI‑related jobs by 2027”) to capture reader interest.  \n- **Why it matters:** Briefly explain AI’s global impact and why India’s trajectory is uniquely significant.  \n- **Purpose of the post:** Offer an in‑depth look at the forces driving AI growth in India, the sectors being transformed, the ecosystem that supports it, and the challenges that must be addressed.\n\n---\n\n## 2. Historical Context  \n### 2.1 Early Foundations (1990s‑2000s)  \n- Academic research in IITs & IISc, early government initiatives (e.g., National Knowledge Network).  \n- First wave of outsourcing and software services that built a talent pool.\n\n### 2.2 The “AI Awakening” (2015‑2020)  \n- Global AI breakthroughs (Deep Learning

In [9]:
print(final_state['outline'])

**Blog Title:**  
*The Rise of Artificial Intelligence in India: Opportunities, Challenges, and the Road Ahead*

---

## 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “India is projected to host 2.2 million AI‑related jobs by 2027”) to capture reader interest.  
- **Why it matters:** Briefly explain AI’s global impact and why India’s trajectory is uniquely significant.  
- **Purpose of the post:** Offer an in‑depth look at the forces driving AI growth in India, the sectors being transformed, the ecosystem that supports it, and the challenges that must be addressed.

---

## 2. Historical Context  
### 2.1 Early Foundations (1990s‑2000s)  
- Academic research in IITs & IISc, early government initiatives (e.g., National Knowledge Network).  
- First wave of outsourcing and software services that built a talent pool.

### 2.2 The “AI Awakening” (2015‑2020)  
- Global AI breakthroughs (Deep Learning, AlphaGo) and their ripple effect in India.  
- Launch of governmen

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CONCEPT: what actually changed from the LCEL chains notebooks, and why it matters
# ─────────────────────────────────────────────────────────────────────────────
#
# This graph does the SAME JOB as one of the earliest LangChain chain files --
# the manual report_gen_chain -> prompt2 | model | parser sequential chain.
# This is that exact pattern, rebuilt in LangGraph's vocabulary:
#
#   LCEL CHAIN                                   LANGGRAPH
#   -------------------------------------------  --------------------------------
#   prompt1 | model | parser | prompt2 | model    two nodes wired by edges
#   output of one step auto-flows to the next     each node reads/writes the
#                                                  SHARED state dict
#   a linear pipe, fixed at definition time        a graph you can branch, loop,
#                                                  or restructure
#
# THE KEY STRUCTURAL IDEA:
# state (BlogState) is SHARED and CUMULATIVE across the whole graph, not passed
# step-to-step like a chain. create_outline writes state['outline']; create_blog
# then READS that same state['outline'] -- every node sees the FULL accumulated
# state so far, not just what the previous node handed it. That is a meaningfully
# different mental model than LCEL's pipe, where each step only sees what the
# IMMEDIATELY PREVIOUS step returned.
#
# Right now, with two nodes wired straight in a line, LangGraph is doing exactly
# what a chain does -- no real advantage yet. The payoff shows up in the NEXT
# kind of file: conditional edges (routing to different nodes based on state,
# like a RunnableBranch chain) and loops (a node that can send flow back to an
# earlier node -- something a chain structurally cannot do). This two-node file
# is the bridge: same job as chaining, new vocabulary, setting up for graphs
# that actually need to be graphs.
# ─────────────────────────────────────────────────────────────────────────────